In [1]:
import sys
sys.path.append("..")
import math
import torch
from models.gpt import GPT, GPTConfig
model = GPT(GPTConfig())
print(sum(p.numel() for p in model.parameters()))
x = torch.randint(0, 8000, (2, 10))
logits, loss = model(x, x)
print(logits.shape, loss.item(), math.log(8000))

5270016
torch.Size([2, 10, 8000]) 8.686968803405762 8.987196820661973


In [5]:
import json
import numpy as np
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file="../tokenizer/kam_bpe_8k.model")
print(sp.get_piece_size(), sp.eos_id())

8000 2


In [3]:
import glob
print(glob.glob("../**/kam_bpe_8k.model", recursive=True))

['..\\data\\raw\\kikamba-corpus\\tokenizer_models\\kam_bpe_8k.model']


In [4]:
import shutil
shutil.copy("../data/raw/kikamba-corpus/tokenizer_models/kam_bpe_8k.model", "../tokenizer/")

'../tokenizer/kam_bpe_8k.model'

In [6]:
def read_docs(path):
    text = open(path, encoding="utf-8").read()
    return [d for d in text.split("\n\n") if d.strip()]

train_docs = read_docs("../data/splits/train.txt")
val_docs = read_docs("../data/splits/val.txt")
print(len(train_docs), len(val_docs))

def encode_docs(docs, path):
    ids = []
    for d in docs:
        ids.extend(sp.encode(d))
        ids.append(sp.eos_id())
    arr = np.array(ids, dtype=np.uint16)
    arr.tofile(path)
    return len(arr)

n_train = encode_docs(train_docs, "../data/splits/train_ids.bin")
n_val = encode_docs(val_docs, "../data/splits/val_ids.bin")
val_chars = sum(len(d) for d in val_docs)
print(n_train, n_val, val_chars, val_chars / n_val)

check = np.fromfile("../data/splits/val_ids.bin", dtype=np.uint16)
print(len(check), sp.decode(check[:30].tolist()))

json.dump({"val_tokens": n_val, "val_chars": val_chars},
          open("../results/val_counts.json", "w"), indent=2)

43613 2295
2627651 219943 783181 3.5608362166561336
219943 Ũtonya kwĩka ata nĩ kana ũsũvĩe syana syaku iikese kwĩtanĩthya na maũndũ mathũku? na jane kya hua mp3 5:39
